# 📦 E-Commerce Order Fulfillment — Phase 1: EDA & Preprocessing
**Dataset:** E-Commerce Shipping & Delivery Performance Dataset (50,000 Records)  
**Source:** Kaggle  
**Target Variable:** `Delivery_Status` (Delivered / Delayed / Returned)  
**Task:** Multi-class Classification — Predict delivery outcome

---
## Table of Contents
1. [Problem Definition & Dataset Understanding](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Preprocessing](#3)
4. [Feature Engineering](#4)
5. [Final Dataset Summary](#5)

In [ ]:
# ============================================================
# CELL 1 — Import all required libraries
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
PALETTE = {'Delivered': '#2ecc71', 'Delayed': '#e74c3c', 'Returned': '#f39c12'}

print('✅ Libraries imported successfully.')

<a id='1'></a>
---
## Section 1 — Problem Definition & Dataset Understanding

### 1.1 Business Problem
E-commerce companies lose revenue and customer trust through late or returned shipments.  
**Goal:** Build a classifier that predicts `Delivery_Status` — whether an order will be **Delivered on time**, **Delayed**, or **Returned** — using order-level features such as region, shipping mode, cost, and time-to-ship.

### 1.2 Why this matters
- Proactively flagging high-risk orders enables logistics teams to intervene early  
- Understanding delay drivers improves supply chain efficiency  
- Reducing returns decreases reverse-logistics costs

### 1.3 ML Formulation
| Item | Detail |
|------|--------|
| Type | Multi-class Classification |
| Target | `Delivery_Status` (3 classes) |
| Features | 9 raw columns → engineered to ~14 |
| Metric | Accuracy, F1-macro, Confusion Matrix |

In [ ]:
# ============================================================
# CELL 2 — Load the dataset and inspect basic structure
# ============================================================

# Load data (update path if running locally)
df = pd.read_csv('E-Commerce_Order_Fulfillment_Dataset__50K_Records_.csv')

print('=== DATASET SHAPE ===')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')

print('\n=== FIRST 5 ROWS ===')
df.head()

In [ ]:
# ============================================================
# CELL 3 — Column data types and memory usage
# ============================================================
print('=== COLUMN INFO ===')
df.info()

print('\n=== DATA TYPES SUMMARY ===')
print(df.dtypes.value_counts())

In [ ]:
# ============================================================
# CELL 4 — Missing values check
# ============================================================
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

print('=== MISSING VALUES ===')
print(missing_df)
print(f'\n✅ Total missing cells: {missing.sum()} — Dataset is COMPLETE.')

In [ ]:
# ============================================================
# CELL 5 — Duplicate rows check
# ============================================================
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

# Check Order_ID uniqueness
print(f'Unique Order_IDs: {df["Order_ID"].nunique():,} out of {len(df):,} rows')
print('✅ All Order_IDs are unique — no duplicate orders.')

In [ ]:
# ============================================================
# CELL 6 — Statistical summary of numeric columns
# ============================================================
print('=== NUMERIC COLUMNS — DESCRIPTIVE STATISTICS ===')
df.describe().round(2)

In [ ]:
# ============================================================
# CELL 7 — Categorical column cardinality overview
# ============================================================
cat_cols = df.select_dtypes(include='object').columns.tolist()
print('=== CATEGORICAL COLUMNS ===')
for col in cat_cols:
    vals = df[col].unique()
    print(f'  {col:20s} → {df[col].nunique()} unique: {list(vals)[:8]}')

<a id='2'></a>
---
## Section 2 — Exploratory Data Analysis (EDA)

EDA is the process of visually and statistically interrogating the dataset *before* modeling to:
- Understand distributions and relationships
- Detect anomalies and outliers
- Identify which features separate the target classes
- Guide feature engineering decisions

In [ ]:
# ============================================================
# CELL 8 — EDA 1: Target class distribution
# ============================================================
# Understanding class balance is critical — imbalanced classes
# can cause models to be biased toward the majority class.

status_counts = df['Delivery_Status'].value_counts()
status_pct = (status_counts / len(df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = [PALETTE[s] for s in status_counts.index]
axes[0].bar(status_counts.index, status_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, (v, p) in enumerate(zip(status_counts.values, status_pct.values)):
    axes[0].text(i, v + 200, f'{v:,}\n({p}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Delivery Status — Count Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Order Count')
axes[0].set_ylim(0, 46000)

# Pie chart
axes[1].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Delivery Status — Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Figure 1 — Target Variable: Delivery_Status', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig1_target_distribution.png', bbox_inches='tight')
plt.show()

print('\nClass counts:')
print(pd.concat([status_counts, status_pct.rename('Percentage %')], axis=1))

In [ ]:
# ============================================================
# CELL 9 — EDA 2: Shipping Mode distribution & vs target
# ============================================================
# Does the shipping mode influence delivery outcome?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overall shipping mode count
mode_counts = df['Shipping_Mode'].value_counts()
sns.barplot(x=mode_counts.index, y=mode_counts.values, palette='Blues_d', ax=axes[0])
for i, v in enumerate(mode_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Orders by Shipping Mode', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')

# Right: stacked proportion by Delivery Status
cross = pd.crosstab(df['Shipping_Mode'], df['Delivery_Status'], normalize='index') * 100
cross[['Delivered', 'Delayed', 'Returned']].plot(
    kind='bar', stacked=True, ax=axes[1],
    color=[PALETTE['Delivered'], PALETTE['Delayed'], PALETTE['Returned']],
    edgecolor='white'
)
axes[1].set_title('Delivery Status by Shipping Mode (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('')
axes[1].legend(title='Status', bbox_to_anchor=(1.02, 1))
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Figure 2 — Shipping Mode Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig2_shipping_mode.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 10 — EDA 3: Customer Region analysis
# ============================================================
# Are certain regions more prone to delays or returns?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: region counts
region_counts = df['Customer_Region'].value_counts()
sns.barplot(x=region_counts.index, y=region_counts.values, palette='Set2', ax=axes[0])
for i, v in enumerate(region_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Orders by Customer Region', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')

# Right: stacked proportion
cross_region = pd.crosstab(df['Customer_Region'], df['Delivery_Status'], normalize='index') * 100
cross_region[['Delivered', 'Delayed', 'Returned']].plot(
    kind='bar', stacked=True, ax=axes[1],
    color=[PALETTE['Delivered'], PALETTE['Delayed'], PALETTE['Returned']],
    edgecolor='white'
)
axes[1].set_title('Delivery Status by Region (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('')
axes[1].legend(title='Status', bbox_to_anchor=(1.02, 1))
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Figure 3 — Regional Delivery Performance', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig3_region_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 11 — EDA 4: Product Category analysis
# ============================================================
# Which product categories are associated with more delays?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_counts = df['Product_Category'].value_counts()
sns.barplot(x=cat_counts.index, y=cat_counts.values, palette='Paired', ax=axes[0])
for i, v in enumerate(cat_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold', fontsize=9)
axes[0].set_title('Orders by Product Category', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

cross_cat = pd.crosstab(df['Product_Category'], df['Delivery_Status'], normalize='index') * 100
cross_cat[['Delivered', 'Delayed', 'Returned']].plot(
    kind='bar', stacked=True, ax=axes[1],
    color=[PALETTE['Delivered'], PALETTE['Delayed'], PALETTE['Returned']],
    edgecolor='white'
)
axes[1].set_title('Delivery Status by Product Category (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Status', bbox_to_anchor=(1.02, 1))
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Figure 4 — Product Category Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig4_category_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 12 — EDA 5: Delivery Days distribution
# ============================================================
# Delivery_Days is a key numeric feature — how does it
# differ across delivery outcome classes?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram by status
for status, color in PALETTE.items():
    subset = df[df['Delivery_Status'] == status]['Delivery_Days']
    axes[0].hist(subset, bins=20, alpha=0.6, label=status, color=color, edgecolor='white')
axes[0].set_title('Delivery Days Distribution by Status', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Delivery Days')
axes[0].set_ylabel('Frequency')
axes[0].legend(title='Status')

# Boxplot
order = ['Delivered', 'Delayed', 'Returned']
colors_box = [PALETTE[s] for s in order]
bp = axes[1].boxplot(
    [df[df['Delivery_Status'] == s]['Delivery_Days'] for s in order],
    labels=order, patch_artist=True, notch=False
)
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Delivery Days Boxplot by Status', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Delivery Days')

plt.suptitle('Figure 5 — Delivery Days Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig5_delivery_days.png', bbox_inches='tight')
plt.show()

print('\nDelivery Days — Mean by Status:')
print(df.groupby('Delivery_Status')['Delivery_Days'].mean().round(2))

In [ ]:
# ============================================================
# CELL 13 — EDA 6: Shipping Cost distribution
# ============================================================
# Higher-cost shipments may be prioritized and less likely to delay.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot — shows both distribution shape and summary stats
plot_data = [df[df['Delivery_Status'] == s]['Shipping_Cost'] for s in ['Delivered', 'Delayed', 'Returned']]
parts = axes[0].violinplot(plot_data, showmedians=True)
for i, (pc, color) in enumerate(zip(parts['bodies'], [PALETTE['Delivered'], PALETTE['Delayed'], PALETTE['Returned']])):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
axes[0].set_xticks([1, 2, 3])
axes[0].set_xticklabels(['Delivered', 'Delayed', 'Returned'])
axes[0].set_title('Shipping Cost — Violin Plot by Status', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Shipping Cost ($)')

# Histogram of overall shipping cost
axes[1].hist(df['Shipping_Cost'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(df['Shipping_Cost'].mean(), color='red', linestyle='--', label=f'Mean: ${df["Shipping_Cost"].mean():.0f}')
axes[1].axvline(df['Shipping_Cost'].median(), color='orange', linestyle='--', label=f'Median: ${df["Shipping_Cost"].median():.0f}')
axes[1].set_title('Overall Shipping Cost Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Shipping Cost ($)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('Figure 6 — Shipping Cost Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig6_shipping_cost.png', bbox_inches='tight')
plt.show()

print('\nShipping Cost — Mean by Status:')
print(df.groupby('Delivery_Status')['Shipping_Cost'].mean().round(2))

In [ ]:
# ============================================================
# CELL 14 — EDA 7: Shipping Cost by Shipping Mode
# ============================================================
# Express & Same Day should cost more than Standard.

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=df, x='Shipping_Mode', y='Shipping_Cost',
    order=['Standard', 'Express', 'Same Day'],
    palette='Blues', ax=ax
)
ax.set_title('Figure 7 — Shipping Cost by Shipping Mode', fontsize=14, fontweight='bold')
ax.set_xlabel('Shipping Mode')
ax.set_ylabel('Shipping Cost ($)')
plt.tight_layout()
plt.savefig('fig7_cost_by_mode.png', bbox_inches='tight')
plt.show()

print('Mean Shipping Cost by Mode:')
print(df.groupby('Shipping_Mode')['Shipping_Cost'].mean().round(2))

In [ ]:
# ============================================================
# CELL 15 — EDA 8: Temporal trend — orders over time
# ============================================================
# Do delivery problems cluster around particular time periods?

# Parse dates
df['Order_Date_dt'] = pd.to_datetime(df['Order_Date'])
df['YearMonth'] = df['Order_Date_dt'].dt.to_period('M')

monthly = df.groupby(['YearMonth', 'Delivery_Status']).size().unstack(fill_value=0)
monthly.index = monthly.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
for status, color in PALETTE.items():
    if status in monthly.columns:
        ax.plot(monthly.index, monthly[status], marker='o', markersize=3,
                label=status, color=color, linewidth=1.5)
ax.set_title('Figure 8 — Monthly Order Volume by Delivery Status (2022–2025)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Order Count')
ax.legend(title='Status')
ax.tick_params(axis='x', rotation=45)

# Show every 6th tick to avoid crowding
ticks = monthly.index.tolist()
ax.set_xticks([ticks[i] for i in range(0, len(ticks), 4)])

plt.tight_layout()
plt.savefig('fig8_temporal_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 16 — EDA 9: Correlation heatmap (numeric features)
# ============================================================
# Correlation helps detect linear relationships between numeric
# features and can expose redundant features.

num_df = df[['Shipping_Cost', 'Delivery_Days']].copy()

# Encode target for correlation
status_map = {'Delivered': 0, 'Delayed': 1, 'Returned': 2}
num_df['Delivery_Status_enc'] = df['Delivery_Status'].map(status_map)

fig, ax = plt.subplots(figsize=(7, 5))
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, square=True, linewidths=0.5)
ax.set_title('Figure 9 — Correlation Matrix (Numeric Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('Correlation with Delivery_Status (encoded):')
print(corr['Delivery_Status_enc'].drop('Delivery_Status_enc').round(3))

In [ ]:
# ============================================================
# CELL 17 — EDA 10: Outlier detection using IQR method
# ============================================================
# Outliers can skew model training, especially for distance-based
# models like SVM. We use the IQR fence to detect them.

def detect_outliers_iqr(series, col_name):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    pct = len(outliers) / len(series) * 100
    print(f'  {col_name:20s}: {len(outliers):5} outliers ({pct:.2f}%) | IQR fence: [{lower:.1f}, {upper:.1f}]')
    return lower, upper

print('=== OUTLIER DETECTION (IQR Method) ===')
for col in ['Shipping_Cost', 'Delivery_Days']:
    detect_outliers_iqr(df[col], col)

# Visual
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['Shipping_Cost', 'Delivery_Days']):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(f'Outlier Check — {col}', fontsize=12, fontweight='bold')
    ax.set_ylabel(col)
plt.suptitle('Figure 10 — Outlier Detection via Boxplot', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig10_outliers.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 18 — EDA 11: Pairwise scatterplot (Shipping Cost vs Delivery Days)
# ============================================================
# Scatterplot coloured by status helps visualise class separation
# in feature space — key for understanding SVM feasibility.

sample = df.sample(3000, random_state=42)  # Sample for speed

fig, ax = plt.subplots(figsize=(9, 6))
for status, color in PALETTE.items():
    subset = sample[sample['Delivery_Status'] == status]
    ax.scatter(subset['Delivery_Days'], subset['Shipping_Cost'],
               alpha=0.35, label=status, color=color, s=15)
ax.set_xlabel('Delivery Days', fontsize=12)
ax.set_ylabel('Shipping Cost ($)', fontsize=12)
ax.set_title('Figure 11 — Shipping Cost vs Delivery Days (by Status, n=3,000)',
             fontsize=13, fontweight='bold')
ax.legend(title='Delivery Status')
plt.tight_layout()
plt.savefig('fig11_scatter.png', bbox_inches='tight')
plt.show()

<a id='3'></a>
---
## Section 3 — Preprocessing

Preprocessing transforms raw data into a format suitable for machine learning:
- **Date parsing** — convert strings to datetime objects to extract meaningful features
- **Label encoding** — convert categorical strings to integers
- **Scaling** — normalize numeric features so no single variable dominates (important for SVM)
- **Train/Test split** — evaluate models on unseen data

In [ ]:
# ============================================================
# CELL 19 — Preprocessing Step 1: Parse date columns
# ============================================================
# Converting string dates to datetime objects lets us compute
# time differences (processing time, shipping lag, etc.)

df_proc = df.copy()  # Work on a copy to preserve raw data

date_cols = ['Order_Date', 'Ship_Date', 'Delivery_Date']
for col in date_cols:
    df_proc[col] = pd.to_datetime(df_proc[col])

print('Date columns parsed successfully:')
print(df_proc[date_cols].dtypes)

In [ ]:
# ============================================================
# CELL 20 — Preprocessing Step 2: Remove ID column
# ============================================================
# Order_ID is a unique identifier with no predictive value.
# Including it would cause data leakage and memorization.

df_proc.drop(columns=['Order_ID'], inplace=True)
print('Order_ID removed. Remaining columns:', df_proc.columns.tolist())

In [ ]:
# ============================================================
# CELL 21 — Preprocessing Step 3: Encode categorical features
# ============================================================
# Machine learning algorithms require numeric inputs.
# Label Encoding is appropriate here for ordinal-style categories;
# we will also create one-hot features in Feature Engineering.

cat_features = ['Customer_Region', 'Product_Category', 'Shipping_Mode']
le_dict = {}

for col in cat_features:
    le = LabelEncoder()
    df_proc[col + '_enc'] = le.fit_transform(df_proc[col])
    le_dict[col] = le  # Store encoder for inverse transform later
    print(f'{col:20s} → classes: {list(le.classes_)}')

# Encode target variable
le_target = LabelEncoder()
df_proc['target'] = le_target.fit_transform(df_proc['Delivery_Status'])
print(f'\nTarget classes: {list(le_target.classes_)} → {list(le_target.transform(le_target.classes_))}')

In [ ]:
# ============================================================
# CELL 22 — Preprocessing Step 4: Handle Delivery_Days_YearMonth
# ============================================================
# Drop temporary columns created during EDA that are not features

drop_cols = ['Order_Date_dt', 'YearMonth'] if 'YearMonth' in df_proc.columns else []
df_proc.drop(columns=drop_cols, errors='ignore', inplace=True)

print('Columns after initial preprocessing:')
print(df_proc.columns.tolist())

<a id='4'></a>
---
## Section 4 — Feature Engineering

Feature engineering creates new, more informative variables from the raw columns.  
Good features can dramatically improve model performance even with a simple algorithm.

**Features we will engineer:**
| New Feature | Source | Rationale |
|---|---|---|
| `processing_time` | Ship_Date − Order_Date | How long before the order was dispatched? |
| `order_month` | Order_Date.month | Seasonality patterns |
| `order_dayofweek` | Order_Date.dayofweek | Weekend orders may be slower |
| `order_year` | Order_Date.year | Year-level trends |
| `is_express` | Shipping_Mode == Express | Binary flag for fast shipping |
| `is_same_day` | Shipping_Mode == Same Day | Binary flag for same-day |
| `cost_per_day` | Shipping_Cost / Delivery_Days | Efficiency ratio |
| One-Hot Features | Region, Category, Mode | Better representation than label encoding |

In [ ]:
# ============================================================
# CELL 23 — Feature Engineering 1: Time-based features
# ============================================================
# Processing time: days from order placement to shipment.
# A long processing time may signal supply chain bottlenecks.

df_proc['processing_time'] = (df_proc['Ship_Date'] - df_proc['Order_Date']).dt.days
df_proc['order_month']      = df_proc['Order_Date'].dt.month
df_proc['order_dayofweek']  = df_proc['Order_Date'].dt.dayofweek  # 0=Mon, 6=Sun
df_proc['order_year']       = df_proc['Order_Date'].dt.year
df_proc['ship_month']       = df_proc['Ship_Date'].dt.month

print('=== Processing Time Statistics ===')
print(df_proc['processing_time'].describe().round(2))

print('\nProcessing Time by Status (mean):')
print(df_proc.groupby('Delivery_Status')['processing_time'].mean().round(2))

In [ ]:
# ============================================================
# CELL 24 — Feature Engineering 2: Binary shipping mode flags
# ============================================================
# Binary flags allow the model to easily distinguish shipping
# modes without assuming ordinal ordering.

df_proc['is_express']  = (df_proc['Shipping_Mode'] == 'Express').astype(int)
df_proc['is_same_day'] = (df_proc['Shipping_Mode'] == 'Same Day').astype(int)
df_proc['is_standard'] = (df_proc['Shipping_Mode'] == 'Standard').astype(int)

print('Express orders:', df_proc['is_express'].sum())
print('Same Day orders:', df_proc['is_same_day'].sum())
print('Standard orders:', df_proc['is_standard'].sum())

In [ ]:
# ============================================================
# CELL 25 — Feature Engineering 3: Cost efficiency ratio
# ============================================================
# cost_per_day = Shipping_Cost / Delivery_Days
# Higher values mean more was spent per delivery day → efficiency proxy.

df_proc['cost_per_day'] = (df_proc['Shipping_Cost'] / df_proc['Delivery_Days']).round(2)

print('Cost Per Day Statistics:')
print(df_proc['cost_per_day'].describe().round(2))

print('\nMean Cost Per Day by Status:')
print(df_proc.groupby('Delivery_Status')['cost_per_day'].mean().round(2))

In [ ]:
# ============================================================
# CELL 26 — Feature Engineering 4: One-hot encoding (region + category)
# ============================================================
# One-hot encoding avoids implying order in categorical features.
# drop_first=True avoids the dummy variable trap (multicollinearity).

ohe_features = ['Customer_Region', 'Product_Category']
df_ohe = pd.get_dummies(df_proc[ohe_features], drop_first=True, prefix=ohe_features)

print('One-hot encoded columns created:')
print(df_ohe.columns.tolist())
df_proc = pd.concat([df_proc, df_ohe], axis=1)

In [ ]:
# ============================================================
# CELL 27 — Visualise new features: processing_time
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram of processing_time
for status, color in PALETTE.items():
    subset = df_proc[df_proc['Delivery_Status'] == status]['processing_time']
    axes[0].hist(subset, bins=15, alpha=0.6, label=status, color=color, edgecolor='white')
axes[0].set_title('Processing Time Distribution by Status', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Processing Time (days)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Mean processing time per month
monthly_proc = df_proc.groupby('order_month')['processing_time'].mean()
axes[1].bar(monthly_proc.index, monthly_proc.values, color='steelblue', edgecolor='white')
axes[1].set_title('Avg Processing Time by Order Month', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Avg Processing Days')
axes[1].set_xticks(range(1, 13))

plt.suptitle('Figure 12 — Engineered Feature: Processing Time', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig12_processing_time.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 28 — Visualise: cost_per_day by status
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))
order_list = ['Delivered', 'Delayed', 'Returned']
data = [df_proc[df_proc['Delivery_Status'] == s]['cost_per_day'] for s in order_list]
colors_list = [PALETTE[s] for s in order_list]

bp = ax.boxplot(data, labels=order_list, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], colors_list):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Figure 13 — Cost Per Day by Delivery Status', fontsize=13, fontweight='bold')
ax.set_ylabel('Shipping Cost / Delivery Days')
plt.tight_layout()
plt.savefig('fig13_cost_per_day.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 29 — Build final feature matrix and target vector
# ============================================================
# Select the final set of features to be used in Phase 2 models.
# We use label-encoded categoricals + numeric + engineered features.

FEATURE_COLS = [
    # Original numeric
    'Shipping_Cost', 'Delivery_Days',
    # Label-encoded categoricals
    'Customer_Region_enc', 'Product_Category_enc', 'Shipping_Mode_enc',
    # Time features
    'processing_time', 'order_month', 'order_dayofweek', 'order_year', 'ship_month',
    # Binary flags
    'is_express', 'is_same_day', 'is_standard',
    # Ratio feature
    'cost_per_day',
    # One-hot (region + category)
] + [c for c in df_proc.columns if c.startswith('Customer_Region_') or c.startswith('Product_Category_')]

X = df_proc[FEATURE_COLS]
y = df_proc['target']  # 0=Delayed, 1=Delivered, 2=Returned (alphabetical LabelEncoder)

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')
print(f'Class distribution: {dict(y.value_counts().sort_index())}')
print(f'\nTarget mapping: {dict(zip(le_target.transform(le_target.classes_), le_target.classes_))}')

In [ ]:
# ============================================================
# CELL 30 — Feature Importance Preview (variance-based ranking)
# ============================================================
# Before modeling, we rank features by their coefficient of
# variation as a simple variance-based importance proxy.

var_importance = X.std() / X.mean().abs()  # Coefficient of variation
var_importance = var_importance.dropna().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
var_importance.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.invert_yaxis()
ax.set_title('Figure 14 — Top Features by Coefficient of Variation', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient of Variation')
plt.tight_layout()
plt.savefig('fig14_feature_importance_cv.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 31 — Scale numeric features
# ============================================================
# StandardScaler: zero mean, unit variance.
# Required for SVM (distance-based) and Neural Networks.
# Tree-based models (Decision Tree, XGBoost) are scale-invariant,
# but we scale all for a fair unified comparison.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=FEATURE_COLS)

print('Feature scaling complete.')
print(f'Mean of scaled features (should ≈ 0): {X_scaled.mean().abs().mean():.6f}')
print(f'Std  of scaled features (should ≈ 1): {X_scaled.std().mean():.6f}')

In [ ]:
# ============================================================
# CELL 32 — Train / Test split (stratified 80/20)
# ============================================================
# Stratified split ensures class proportions are maintained
# in both train and test sets — critical for imbalanced data.
# random_state=42 ensures reproducibility.

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Train set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test  set: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)')
print('\nClass distribution — Train:')
print(pd.Series(y_train).value_counts().sort_index())
print('\nClass distribution — Test:')
print(pd.Series(y_test).value_counts().sort_index())

In [ ]:
# ============================================================
# CELL 33 — Save preprocessed data for Phase 2
# ============================================================
# Saving ensures Phase 2 uses exactly the same preprocessed data,
# guaranteeing reproducible and fair model comparisons.

X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

# Save feature column list
import json
with open('feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

print('✅ Preprocessed data saved:')
print('   → X_train.csv, X_test.csv, y_train.csv, y_test.csv')
print('   → feature_columns.json')
print('\nPhase 2 is ready to begin!')

<a id='5'></a>
---
## Section 5 — Final Dataset Summary

| Item | Value |
|---|---|
| Raw rows | 50,000 |
| Missing values | 0 |
| Duplicates | 0 |
| Target classes | 3 (Delivered 80.1%, Delayed 15.0%, Returned 4.9%) |
| Raw features used | 7 (after dropping Order_ID and dates) |
| Engineered features | +7 (processing_time, month, weekday, year, flags, cost_per_day) |
| One-hot encoded | +9 columns (region × 4 + category × 5) |
| **Total features** | **~23** |
| Train / Test split | 80% / 20% (stratified) |
| Scaling | StandardScaler |

### Key EDA Findings
1. **Class imbalance** — 80% Delivered vs 5% Returned; models need F1-macro evaluation
2. **Delivery_Days** is the strongest separator between Delivered and Delayed
3. **Shipping_Mode** affects delivery rate slightly but is not strongly separating
4. **Processing time** varies by status — delayed orders tend to ship later
5. **Cost per day** is higher for delivered orders — suggesting faster, pricier shipping succeeds
6. Regional differences are minor — all 5 regions are roughly equally represented
7. No outliers severe enough to require removal; the IQR bounds are within realistic ranges

### Ready for Phase 2
- Logistic Regression (Baseline)
- Support Vector Machine (SVM)
- XGBoost (Gradient Boosting)